In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import os
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display

In [14]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [15]:
file_path = os.path.join("..", "data", "processed", "feature_select.csv")

# Read CSV with index_col=0 to avoid unnamed column issues
data = pd.read_csv(file_path, index_col=0)
data

,Enrollment_ID,Study_Prog_Exam_Completed,Prior_Edu_Country,Study_Prog_Group_Size,Student_Enrollment_Gap,Exit_Status,Study_Prog_Name_Accountancy,Study_Prog_Name_Allied_Medical_Care,Study_Prog_Name_Arts_Therapies,Study_Prog_Name_Bachelor_of_Law,...,Prior_Edu_Postcode_7414_AR,Prior_Edu_Postcode_7553_VZ,Prior_Edu_Postcode_8017_CA,Prior_Edu_Postcode_8024_AH,Prior_Edu_Postcode_8031_AA,Prior_Edu_Postcode_8081_VV,Prior_Edu_Postcode_8233_GA,Prior_Edu_Postcode_8932_NX,Prior_Edu_Postcode_9723_ZS,Prior_Edu_Postcode_Others
0,1546413,1,1,347,164.0,leaving_unsuccessfully,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1549026,0,1,347,68.0,leaving_unsuccessfully,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,1543665,1,1,347,49.0,leaving_unsuccessfully,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,1556941,0,1,347,11.0,leaving_unsuccessfully,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,1536214,1,1,347,63.0,leaving_unsuccessfully,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48386,1964301,0,1,1663,51.0,leaving_unsuccessfully,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
48387,1966882,0,1,1663,51.0,leaving_unsuccessfully,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
48388,1972247,1,0,1663,-16.0,leaving_successfully,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
48389,1973119,0,1,1663,99.0,leaving_unsuccessfully,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
data.columns

Index(['Enrollment_ID', 'Study_Prog_Exam_Completed', 'Prior_Edu_Country',
       'Study_Prog_Group_Size', 'Student_Enrollment_Gap', 'Exit_Status',
       'Study_Prog_Name_Accountancy', 'Study_Prog_Name_Allied_Medical_Care',
       'Study_Prog_Name_Arts_Therapies', 'Study_Prog_Name_Bachelor_of_Law',
       ...
       'Prior_Edu_Postcode_7414_AR', 'Prior_Edu_Postcode_7553_VZ',
       'Prior_Edu_Postcode_8017_CA', 'Prior_Edu_Postcode_8024_AH',
       'Prior_Edu_Postcode_8031_AA', 'Prior_Edu_Postcode_8081_VV',
       'Prior_Edu_Postcode_8233_GA', 'Prior_Edu_Postcode_8932_NX',
       'Prior_Edu_Postcode_9723_ZS', 'Prior_Edu_Postcode_Others'],
      dtype='object', length=300)

In [17]:
print(data['Exit_Status'].value_counts())

Exit_Status
leaving_unsuccessfully    43154
switching_internally       3489
leaving_successfully       1748
Name: count, dtype: int64


In [18]:
# Define a mapping for Exit_Status
exit_status_mapping = {
    'leaving_successfully': 0,
    'switching_internally': 1,
    'leaving_unsuccessfully': 2
}

# Apply the mapping to replace categorical values with numbers
data['Exit_Status'] = data['Exit_Status'].map(exit_status_mapping)

# Show the first few rows to verify
print(data['Exit_Status'].value_counts())


Exit_Status
2    43154
1     3489
0     1748
Name: count, dtype: int64


In [19]:
print(data['Exit_Status'].value_counts())

Exit_Status
2    43154
1     3489
0     1748
Name: count, dtype: int64


In [20]:
# From the last result from the EDA.ipynb, we saw that Prior_Edu_Postcode and Prior_Edu_Country have a corr >0.7, therefore we can drop them here.
# However, we are dropping them from the already one-hot-encoded data called feature_select

# Drop columns that start with 'Prior_Edu_Postcode' or 'Prior_Edu_Country'
cols_to_drop = [col for col in data.columns if col.startswith("Prior_Edu_Postcode") or col.startswith("Prior_Edu_Country")]

# Print columns to be removed
print("Columns to be removed:")
print(cols_to_drop)

# Drop the columns
data = data.drop(columns=cols_to_drop)

# Verify the remaining columns
print("Remaining columns:")
print(data.columns)



Columns to be removed:
['Prior_Edu_Country', 'Prior_Edu_Postcode_0000_ABROAD', 'Prior_Edu_Postcode_1033_SB', 'Prior_Edu_Postcode_1071_ED', 'Prior_Edu_Postcode_1078_VN', 'Prior_Edu_Postcode_1097_DZ', 'Prior_Edu_Postcode_1102_CV', 'Prior_Edu_Postcode_1211_EN', 'Prior_Edu_Postcode_1213_VB', 'Prior_Edu_Postcode_1215_PN', 'Prior_Edu_Postcode_1217_GH', 'Prior_Edu_Postcode_1217_PM', 'Prior_Edu_Postcode_1251_GB', 'Prior_Edu_Postcode_1273_JP', 'Prior_Edu_Postcode_1276_KD', 'Prior_Edu_Postcode_1325_PL', 'Prior_Edu_Postcode_1326_AA', 'Prior_Edu_Postcode_1333_VD', 'Prior_Edu_Postcode_1334_PA', 'Prior_Edu_Postcode_1382_CD', 'Prior_Edu_Postcode_1401_RT', 'Prior_Edu_Postcode_1405_HM', 'Prior_Edu_Postcode_1406_NR', 'Prior_Edu_Postcode_1422_MT', 'Prior_Edu_Postcode_1507_EK', 'Prior_Edu_Postcode_1817_BC', 'Prior_Edu_Postcode_1902_GT', 'Prior_Edu_Postcode_2012_BL', 'Prior_Edu_Postcode_2012_MN', 'Prior_Edu_Postcode_2012_WT', 'Prior_Edu_Postcode_2321_KS', 'Prior_Edu_Postcode_2334_CP', 'Prior_Edu_Postcode_2

Print only the pairs of columns with correlation above 0.7

In [21]:
# Convert Exit_Status to numeric values automatically (if not already done)
data['Exit_Status'] = data['Exit_Status'].astype('category').cat.codes  # This will assign 0,1,2 automatically

# Drop Enrollment_ID (since it's just an identifier)
data_numeric = data.drop(columns=['Enrollment_ID'])

# Compute correlation matrix
corr_matrix = data_numeric.corr()

# Find pairs with correlation above 0.7 (excluding self-correlation)
high_corr_pairs = []
threshold = 0.7

for col1 in corr_matrix.columns:
    for col2 in corr_matrix.columns:
        if col1 != col2 and corr_matrix.loc[col1, col2] > threshold:
            high_corr_pairs.append((col1, col2, corr_matrix.loc[col1, col2]))

# Remove duplicate pairs (since correlation is symmetric)
unique_high_corr_pairs = set(tuple(sorted(pair[:2])) + (pair[2],) for pair in high_corr_pairs)

# Print highly correlated column pairs
print("Highly Correlated Column Pairs (Above 70%):")
for col1, col2, corr in unique_high_corr_pairs:
    print(f"{col1} - {col2}: {corr:.2f}")


Highly Correlated Column Pairs (Above 70%):
Prior_Edu_School_Location_0000_ABROAD - Prior_Edu_Type_BUITENL_SL: 0.87


Iteratively remove one of the columns from the highly correlated pairs until all remaining columns have a correlation of less than 0.7

In [22]:
# Drop Enrollment_ID (since it's just an identifier)
data_numeric = data.drop(columns=['Enrollment_ID'])

# Compute correlation matrix
corr_matrix = data_numeric.corr()

# Threshold for high correlation
threshold = 0.7

# Identify pairs of highly correlated columns
high_corr_pairs = set()
for col1 in corr_matrix.columns:
    for col2 in corr_matrix.columns:
        if col1 != col2 and corr_matrix.loc[col1, col2] > threshold:
            high_corr_pairs.add((col1, col2))

# Identify columns to remove to reduce correlation
columns_to_remove = set()
for col1, col2 in high_corr_pairs:
    if col1 not in columns_to_remove and col2 not in columns_to_remove:
        columns_to_remove.add(col2)  # Remove one of the highly correlated columns

# Keep only uncorrelated columns (less than 0.7 correlation)
final_columns = [col for col in data_numeric.columns if col not in columns_to_remove]
data_final = data_numeric[final_columns]

# Show the remaining columns and the first few rows
print("Remaining Columns After Removing High Correlation:")
print(data_final.columns)
print("\nData Preview:")
data_final


Remaining Columns After Removing High Correlation:
Index(['Study_Prog_Exam_Completed', 'Study_Prog_Group_Size',
       'Student_Enrollment_Gap', 'Exit_Status', 'Study_Prog_Name_Accountancy',
       'Study_Prog_Name_Allied_Medical_Care', 'Study_Prog_Name_Arts_Therapies',
       'Study_Prog_Name_Bachelor_of_Law',
       'Study_Prog_Name_Biology_Medical_Laboratory_Research',
       'Study_Prog_Name_Built_Environment',
       ...
       'Prior_Edu_School_Location_WEESP',
       'Prior_Edu_School_Location_WIJK_BIJ_DUURSTEDE',
       'Prior_Edu_School_Location_WOERDEN',
       'Prior_Edu_School_Location_ZAANDAM',
       'Prior_Edu_School_Location_ZALTBOMMEL',
       'Prior_Edu_School_Location_ZEIST', 'Prior_Edu_School_Location_ZEVENAAR',
       'Prior_Edu_School_Location_ZOETERMEER',
       'Prior_Edu_School_Location_ZUTPHEN',
       'Prior_Edu_School_Location_ZWOLLE'],
      dtype='object', length=142)

Data Preview:


,Study_Prog_Exam_Completed,Study_Prog_Group_Size,Student_Enrollment_Gap,Exit_Status,Study_Prog_Name_Accountancy,Study_Prog_Name_Allied_Medical_Care,Study_Prog_Name_Arts_Therapies,Study_Prog_Name_Bachelor_of_Law,Study_Prog_Name_Biology_Medical_Laboratory_Research,Study_Prog_Name_Built_Environment,...,Prior_Edu_School_Location_WEESP,Prior_Edu_School_Location_WIJK_BIJ_DUURSTEDE,Prior_Edu_School_Location_WOERDEN,Prior_Edu_School_Location_ZAANDAM,Prior_Edu_School_Location_ZALTBOMMEL,Prior_Edu_School_Location_ZEIST,Prior_Edu_School_Location_ZEVENAAR,Prior_Edu_School_Location_ZOETERMEER,Prior_Edu_School_Location_ZUTPHEN,Prior_Edu_School_Location_ZWOLLE
0,1,347,164.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,347,68.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,347,49.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,347,11.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,347,63.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48386,0,1663,51.0,2,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
48387,0,1663,51.0,2,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
48388,1,1663,-16.0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
48389,0,1663,99.0,2,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


We see 142 columns left out of 300

In [23]:
data_final.to_csv(os.path.join("..", "data", "processed", "removed_high_corr.csv"), header=True)

In [24]:
data = data_final
data

,Study_Prog_Exam_Completed,Study_Prog_Group_Size,Student_Enrollment_Gap,Exit_Status,Study_Prog_Name_Accountancy,Study_Prog_Name_Allied_Medical_Care,Study_Prog_Name_Arts_Therapies,Study_Prog_Name_Bachelor_of_Law,Study_Prog_Name_Biology_Medical_Laboratory_Research,Study_Prog_Name_Built_Environment,...,Prior_Edu_School_Location_WEESP,Prior_Edu_School_Location_WIJK_BIJ_DUURSTEDE,Prior_Edu_School_Location_WOERDEN,Prior_Edu_School_Location_ZAANDAM,Prior_Edu_School_Location_ZALTBOMMEL,Prior_Edu_School_Location_ZEIST,Prior_Edu_School_Location_ZEVENAAR,Prior_Edu_School_Location_ZOETERMEER,Prior_Edu_School_Location_ZUTPHEN,Prior_Edu_School_Location_ZWOLLE
0,1,347,164.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,347,68.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,347,49.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,347,11.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,347,63.0,2,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48386,0,1663,51.0,2,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
48387,0,1663,51.0,2,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
48388,1,1663,-16.0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
48389,0,1663,99.0,2,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
print(data['Exit_Status'].value_counts())

Exit_Status
2    43154
1     3489
0     1748
Name: count, dtype: int64


# Run a stats test to check for significance

In [26]:
data.columns

Index(['Study_Prog_Exam_Completed', 'Study_Prog_Group_Size',
       'Student_Enrollment_Gap', 'Exit_Status', 'Study_Prog_Name_Accountancy',
       'Study_Prog_Name_Allied_Medical_Care', 'Study_Prog_Name_Arts_Therapies',
       'Study_Prog_Name_Bachelor_of_Law',
       'Study_Prog_Name_Biology_Medical_Laboratory_Research',
       'Study_Prog_Name_Built_Environment',
       ...
       'Prior_Edu_School_Location_WEESP',
       'Prior_Edu_School_Location_WIJK_BIJ_DUURSTEDE',
       'Prior_Edu_School_Location_WOERDEN',
       'Prior_Edu_School_Location_ZAANDAM',
       'Prior_Edu_School_Location_ZALTBOMMEL',
       'Prior_Edu_School_Location_ZEIST', 'Prior_Edu_School_Location_ZEVENAAR',
       'Prior_Edu_School_Location_ZOETERMEER',
       'Prior_Edu_School_Location_ZUTPHEN',
       'Prior_Edu_School_Location_ZWOLLE'],
      dtype='object', length=142)

In [ ]:
# Drop Enrollment_ID if present
if 'Enrollment_ID' in data.columns:
    data = data.drop(columns=['Enrollment_ID'])

# Ensure Exit_Status is numeric (already mapped previously)
y = data['Exit_Status']

# Use all other columns as predictors (X)
X = data.drop(columns=['Exit_Status'])

# Add constant term for intercept
X = sm.add_constant(X)

# Fit OLS model
model = sm.OLS(y, X).fit()

# Print summary
print(model.summary())

# Extract statistically significant features (p-value < 0.05)
significant_features = model.pvalues[model.pvalues < 0.05].index
print("\nSignificant Features (p < 0.05):")
print(significant_features)

# Extract non-significant features (p-value >= 0.05)
non_significant_features = model.pvalues[model.pvalues >= 0.05].index
print("\nNon-Significant Features (p >= 0.05):")
print(non_significant_features)


                            OLS Regression Results                            
Dep. Variable:            Exit_Status   R-squared:                       0.070
Model:                            OLS   Adj. R-squared:                  0.067
Method:                 Least Squares   F-statistic:                     26.02
Date:                Sat, 29 Mar 2025   Prob (F-statistic):               0.00
Time:                        12:06:36   Log-Likelihood:                -27455.
No. Observations:               48391   AIC:                         5.519e+04
Df Residuals:                   48251   BIC:                         5.642e+04
Df Model:                         139                                         
Covariance Type:            nonrobust                                         
                                                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

Highly statisfically significant features on Exiting include: 
Study_Prog_Exam_Completed
Study_Prog_Group_Size
Student_Enrollment_Gap
Study_Prog_Name( 19 features from it)
Prior_Edu_School_Location (22 features from it)

while the non significant features on Exiting include: 
Prior_Edu_Type
Study_Prog_Name
Prior_Edu_School_Location (more than 50 features from it)

Ambigious features: 
Study_Prog_Name
Prior_Edu_School_Location  

In [39]:
# Set options to display all rows and columns
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.max_colwidth', None)    # Show full content of each column
pd.set_option('display.width', None)           # Adjust width for better readability
pd.set_option('display.expand_frame_repr', True)  # Expand the DataFrame across multiple lines if necessary
